# sbi-xray-calibration: walkthrough

When can you trust an SBI posterior for an X-ray spectrum, how do you detect when
you can't, and how much can recalibration repair?

This notebook walks the whole story end-to-end. It never retrains a flow or
re-runs a heavy benchmark, but it reads two kinds of input:

- **Committed with the repo** (work on a fresh clone): the analysis records
  (`outputs/calibration/*/summary.json`, `outputs/detect/results.jsonl`,
  `outputs/gonogo/summary.jsonl`) and the figures under `outputs/diagnostics/`
  and `outputs/calibration/`.
- **Gitignored, regenerate first** (needed by sections 1, 2 and the ΔΓ table in
  section 4): the training sets `data/sim/*.npz`
  (`python -m sbixcal.simulate --config configs/sim_modelA_prod_train.yaml`),
  the trained checkpoints in `outputs/models/`
  (`python scripts/run_train_npe.py --config configs/train_npe_prod.yaml`),
  and `outputs/detect/consequence.jsonl`
  (`python scripts/run_detect_benchmark.py --config configs/detect.yaml`).
  One `python scripts/run_all.py` regenerates everything, stage by stage,
  skip-if-exists.

With those in place the notebook cold-loads the production NPE flows and reads
the finished calibration / detection / NS outputs; runtime is a few seconds.

Order: example spectra per count level, a live NPE posterior demo, the
calibration story (one flow was miscalibrated; the checks caught it; a
GO/NO-GO pass shows it is a single-flow training artifact, with the count regime ruled out),
the detection ROC grid and the Fe-K-line ΔΓ silent-failure cost, then the summary
(including the NS speed-vs-trust and the gain-shift-flagged-by-evidence result).


In [ ]:
import os, sys, json
import numpy as np
import matplotlib.pyplot as plt

os.environ.setdefault("OMP_NUM_THREADS", "4")

# repo paths (notebook lives in notebooks/)
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if os.path.basename(os.getcwd()) != "notebooks":
    # if executed from repo root, adjust
    ROOT = os.path.abspath(os.getcwd())
SRC = os.path.join(ROOT, "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)

OUT = os.path.join(ROOT, "outputs")
DATA = os.path.join(ROOT, "data", "sim")
LEVELS = ["faint", "medium", "bright"]
print("repo root:", ROOT)
print("torch / sbi import ...")
import torch
from sbixcal import train_npe
print("ok")

## 1. Example spectra per count level

The three count regimes (~100 / 1000 / 10000 median total counts) are the same
absorbed power-law + blackbody Model A folded through the bundled XMM-Newton
EPIC-pn response, at three effective exposures. We pull a few already-simulated
spectra from the production training sets in `data/sim/` (gitignored; regenerate
with the simulate config named in the header) -- Poisson counts vs channel
energy.


In [ ]:
def load_sim(level):
    p = os.path.join(DATA, f"modelA_prod_train_{level}.npz")
    d = np.load(p, allow_pickle=True)
    return d

emid = None
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), sharey=False)
for ax, level in zip(axes, LEVELS):
    d = load_sim(level)
    e_min, e_max = d["e_min"], d["e_max"]
    emid = 0.5 * (e_min + e_max)
    x = d["x"]
    rng = np.random.default_rng(0)
    idx = rng.choice(len(x), size=4, replace=False)
    for i in idx:
        ax.step(emid, x[i] + 1e-9, where="mid", lw=1.2, alpha=0.8)
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel("energy [keV]")
    ax.set_title(f"{level}  (~{float(d['median_total_counts']):.0f} counts)")
axes[0].set_ylabel("counts / channel")
fig.suptitle("Example EPIC-pn spectra across the three count regimes (Model A)")
fig.tight_layout()
plt.show()

## 2. A live NPE posterior demo (cold-loaded flow)

`train_npe.load_posterior(<dir>)` rebuilds the exact NSF + 1-D-CNN architecture
from `arch.json` and loads the trained weights, no training data needed. We take
one held-out medium-count spectrum, run the amortized flow on it (milliseconds),
and compare the posterior marginals to the truth that generated it.


In [ ]:
level = "medium"
mdir = os.path.join(OUT, "models", f"train_npe_prod_{level}")
posterior, info = train_npe.load_posterior(mdir, device="cpu")
print("loaded:", os.path.basename(mdir))
print("params:", info["param_names"], "| ~counts:", info["median_total_counts"])

d = load_sim(level)
# use a spectrum from the END of the file (the flow trained on a 50k set; this is
# just a demonstration draw, not a held-out-set claim).
j = -7
x_obs = torch.tensor(d["x"][j], dtype=torch.float32)
theta_true = d["theta"][j]

samples = posterior.sample((4000,), x=x_obs, show_progress_bars=False).numpy()
pnames = info["param_names"]

fig, axes = plt.subplots(1, 5, figsize=(15, 2.8))
for k, ax in enumerate(axes):
    ax.hist(samples[:, k], bins=40, color="#0072B2", alpha=0.8)
    ax.axvline(theta_true[k], color="#D55E00", lw=2, label="truth")
    ax.set_title(pnames[k].replace("_1_", " "), fontsize=9)
    ax.set_yticks([])
axes[0].legend(fontsize=8)
fig.suptitle(f"Amortized NPE posterior on one {level} spectrum (vermilion = truth)")
fig.tight_layout()
plt.show()

print("posterior median vs truth:")
for k, nm in enumerate(pnames):
    print(f"  {nm:22s}  med={np.median(samples[:,k]):.4g}   true={theta_true[k]:.4g}")

## 3. Calibration: catching a miscalibrated flow

One trained production flow (the bright, ~10000-count one) passed every recovery-quality
check yet was miscalibrated: coverage deviation ~0.11, SBC rank histograms
collapsed (∪-shaped) on all five parameters. It had hit its 150-epoch
training-budget cap with a train/val gap. Only the calibration checks caught it:
SBC + a coverage test flagged it, split-conformal recalibration repaired it
(~0.11 → ~0.03), and the IS-refinement low-ESS flag fired on that flow.

A GO/NO-GO robustness pass (next cell) shows it is a **single-flow training
artifact**: three reseeds + an uncapped retrain on the same data
all land near-calibrated (deviation 0.014–0.033). Recovery quality does not certify
calibration, so validate every flow you deploy. The coverage money panel below shows
the one miscalibrated bright flow (raw sags, conformal repairs); we also read the
per-level deviations and SBC p-values from each `summary.json`.


In [ ]:
import matplotlib.image as mpimg
panel = os.path.join(OUT, "diagnostics", "coverage_money_panel.png")
fig, ax = plt.subplots(figsize=(13, 4.6))
ax.imshow(mpimg.imread(panel)); ax.axis("off")
plt.show()

print("per-level calibration summary (read from summary.json):")
print(f"{'level':7s} {'~counts':>8s} {'raw |dev|':>10s} {'recal |dev|':>12s} "
      f"{'IS low-ESS':>11s}")
for level in LEVELS:
    s = json.load(open(os.path.join(OUT, "calibration", level, "summary.json")))
    cov = s["coverage"]; isr = s["is_refinement"]
    print(f"{level:7s} {float(s['median_total_counts']):8.0f} "
          f"{cov['raw_mean_abs_dev']:10.3f} {cov['recal_mean_abs_dev']:12.3f} "
          f"{isr['cov_testset_low_ess_fraction']:10.0%}")

In [ ]:
# SBC rank histograms per level (committed figures): uniform = calibrated,
# U-shaped = over-confident. Note the bright panel -- this is the ONE flow that
# was miscalibrated (it hit its epoch cap); the GO/NO-GO cell below shows
# reseeds/uncapped retrains of the same bright data come out uniform.
fig, axes = plt.subplots(1, 3, figsize=(14, 3.4))
for ax, level in zip(axes, LEVELS):
    img = mpimg.imread(os.path.join(OUT, "calibration", level, "sbc_ranks.png"))
    ax.imshow(img); ax.axis("off"); ax.set_title(level)
fig.suptitle("SBC rank histograms (uniform = calibrated; U-shaped bright = the miscalibrated flow)")
fig.tight_layout()
plt.show()

### GO/NO-GO robustness, the over-confidence is a single-flow artifact

The same bright (~10000-count) training data, reseeded three times and retrained
once with the epoch cap lifted (150 → 400). If high counts meant over-confident
posteriors by default, every variant would over-cover. Instead they all land
near-calibrated (raw coverage deviation 0.014–0.033 vs the original flow's
~0.11), and the cleanest control, the uncapped retrain, converges at 162 epochs
to deviation 0.014 with uniform SBC. The original flow was undertrained, not in a
bad regime. Read straight from `outputs/gonogo/summary.jsonl`.


In [ ]:
# GO/NO-GO robustness table: the bright production flow vs reseeds + uncapped retrain.
gonogo = [json.loads(l) for l in open(os.path.join(OUT, "gonogo", "summary.jsonl"))]
cal = [r for r in gonogo if r["kind"] == "calibration"]

print("bright-level flows trained on the SAME ~10000-count data (differ by seed / epoch cap):")
print(f"{'variant':18s} {'epochs/cap':>12s} {'raw cov dev':>12s} {'conf cov dev':>13s} {'SBC ks_p_min':>13s}")
# original production flow (the miscalibrated one), from its calibration summary
prod = json.load(open(os.path.join(OUT, "calibration", "bright", "summary.json")))
pcov = prod["coverage"]
print(f"{'production (orig)':18s} {'150/150':>12s} {pcov['raw_mean_abs_dev']:12.3f} "
      f"{pcov['recal_mean_abs_dev']:13.3f} {'~0 (5/5 fail)':>13s}")
for r in sorted(cal, key=lambda r: r["variant"]):
    cap = f"{r['epochs_trained']}/{r['max_num_epochs']}"
    print(f"{r['variant']:18s} {cap:>12s} {r['cov_dev_raw']:12.3f} "
          f"{r['cov_dev_conformal']:13.3f} {r['sbc_ks_p_min']:13.2e}")

devs = [r["cov_dev_raw"] for r in cal]
print(f"\nrobustness-variant raw coverage deviation range: {min(devs):.3f}-{max(devs):.3f}")
print("=> the original 0.113 is the outlier; the over-confidence was an epoch-cap / "
      "undertraining artifact, NOT the high-count regime.")
print("   The trust toolkit (SBC + coverage) caught the bad flow; conformal repaired it.")

# detection spot-check is seed-robust (read the seed101 spot rows)
spot = [r for r in gonogo if r["kind"] == "detect_spot"]
if spot:
    print("\ndetection spot-check on the seed-101 flow (seed-robust):")
    for r in spot:
        print(f"  {r['family']} strength {r['strength']:g}  {r['detector']} AUC "
              f"{r['auc']:.3f}  (expected ~{r['expected_auc']})")

## 4. Detection, the ROC grid and the silent-failure cost

A 144-cell ROC grid (4 misspecification families × strength
grids × 3 count levels × 3 detectors). The heatmap shows the best AUC per
(detector, family) per level. D1/PPC catches the Fe-K line (B1), D2 leads on partial-covering
(B2) among per-spectrum detectors, D3/marginal-C2ST separates the populations for B2
and B3 but is a supervised two-sample statistic , a population measure separate from the per-spectrum trust scores,
and the gain-shift family (B4) is invisible to every detector (the whole B4 column sits
at chance).


In [ ]:
grid_img = os.path.join(OUT, "diagnostics", "detector_auc_grid.png")
fig, ax = plt.subplots(figsize=(14, 4.4))
ax.imshow(mpimg.imread(grid_img)); ax.axis("off")
plt.show()

# best AUC per (level, family) straight from results.jsonl
res = [json.loads(l) for l in open(os.path.join(OUT, "detect", "results.jsonl"))]
fams = ["B1", "B2", "B3", "B4"]
print("best AUC per family per level (best detector per family):")
print(f"{'level':7s} " + " ".join(f"{f:>16s}" for f in fams))
for level in LEVELS:
    cells = []
    for fam in fams:
        sub = [r for r in res if r["level"] == level and r["family"] == fam]
        b = max(sub, key=lambda r: r["auc"])
        cells.append(f"{b['auc']:.2f} ({b['detector']})")
    print(f"{level:7s} " + " ".join(f"{c:>16s}" for c in cells))

### The silent-failure cost: an unmodeled Fe-K line biases Γ

Where the line slips past the detector it biases the inferred photon
index. The signed bias grows with counts; at the strongest line it pulls Γ softer
by **+0.20 on average (median +0.26) at bright**. That's the "undetected and
biased" number that matters for a downstream measurement.


In [ ]:
dg_img = os.path.join(OUT, "diagnostics", "dgamma_silent_failure.png")
fig, ax = plt.subplots(figsize=(7.5, 5.2))
ax.imshow(mpimg.imread(dg_img)); ax.axis("off")
plt.show()

cons = [json.loads(l) for l in open(os.path.join(OUT, "detect", "consequence.jsonl"))]
print("B1 Gamma-bias at the STRONGEST line (norm 3e-4):")
print(f"{'level':7s} {'<dGamma>':>10s} {'median':>9s} {'<|dGamma|>':>11s}")
for level in LEVELS:
    r = [c for c in cons if c["level"] == level and abs(c["strength"] - 3e-4) < 1e-12][0]
    print(f"{level:7s} {r['dGamma_bias_mean']:+10.3f} {r['dGamma_bias_median']:+9.3f} "
          f"{r['abs_bias_mean']:11.3f}")

## 5. Summary

- **Calibration:** one production flow passed every recovery check yet was
  miscalibrated (|dev| ~0.11, SBC collapsed) because it had hit its epoch cap. The
  trust toolkit caught it (SBC + coverage), split-conformal repaired it
  (~0.11 -> ~0.03), and the IS-refinement low-ESS flag fired on that flow.
  Reseeds + an uncapped retrain on the same data bring the coverage back
  (|dev| 0.014-0.033), so the over-confidence is a single-flow training artifact;
  a rank-level SBC non-uniformity on the power-law norm persists at high counts.
  Recovery quality does not certify calibration, so validate every flow.
- **Detection:** the Fe-K line (D1/PPC, AUC 0.97 at >=1000 ct) and partial-covering
  (D2 among per-spectrum detectors; D3 population-separability up to 0.96) are
  catchable; the wrong continuum (B3) is weak and only the population test sees it;
  **detector gain shifts up to 3% are undetectable by all three per-spectrum trust
  scores at every count level (AUC ~ 0.50)**; no score catches it, an open problem.
- **Consequence:** an undetected Fe-K line biases Gamma softer by up to +0.20 mean
  (+0.26 median). The misspecifications that are hardest to detect are the
  ones that bias a downstream measurement.
- **Speed vs trust (nested sampling):** NS on the same Poisson likelihood is
  **~8 800-13 000x slower per spectrum** than the amortized NPE and quantile-equivalent
  where the flow is calibrated (0.04-0.10 of the prior width). Its Bayesian evidence
  catches the Fe-K line strongly (dlogZ ~ -890 at bright) but, once counts are
  controlled, does not separate the 3% gain shift from clean data either (a paired
  test gives dlogZ consistent with zero). NS earns its cost on the line and on the
  coverage guarantee.